# Студенческая библиотека — итоговый проект (лабораторные №2–10)

**Выполнить все**: создаёт Python-файлы, демонстрационную SQLite-базу, показывает результаты запросов, экспортирует отчёты и запускает тесты. Лабораторная №1 (WEEEK) выполняется отдельно. GitHub-операции требуют реального репозитория.


In [ ]:
import os, pathlib
print("Текущая папка:", os.getcwd())


In [ ]:
from pathlib import Path
Path('library_functions.py').write_text('"""Validation and formatting helpers used by the library application."""\nfrom datetime import date\n\n\ndef validate_year(year):\n    if type(year) is not int or not 1000 <= year <= 2100:\n        raise ValueError(\'Год должен быть целым числом от 1000 до 2100\')\n    return True\n\n\ndef validate_quantity(quantity):\n    if type(quantity) is not int or quantity <= 0:\n        raise ValueError(\'Количество должно быть положительным целым числом\')\n    return True\n\n\ndef validate_student_card(student_card):\n    if not isinstance(student_card, str) or not student_card.strip() or len(student_card) > 20:\n        raise ValueError(\'Номер билета: от 1 до 20 символов\')\n    return True\n\n\ndef calculate_overdue_days(due_date, return_date):\n    try:\n        due = date.fromisoformat(due_date)\n        returned = date.fromisoformat(return_date)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(\'Даты должны быть в формате YYYY-MM-DD\') from exc\n    return max(0, (returned - due).days)\n\n\ndef format_book_title(title, max_length=30):\n    if not isinstance(title, str) or not title.strip():\n        raise ValueError(\'Название не может быть пустым\')\n    if type(max_length) is not int or max_length < 3:\n        raise ValueError(\'Максимальная длина должна быть не меньше 3\')\n    return title if len(title) <= max_length else title[:max_length - 3] + \'...\'\n', encoding="utf-8")
print("Создан library_functions.py")


In [ ]:
from pathlib import Path
Path('database.py').write_text('"""SQLite data access for the student library project."""\nimport sqlite3\nfrom pathlib import Path\n\nDB_PATH = Path(__file__).with_name(\'library.db\')\n\n\ndef get_connection(db_path=None):\n    conn = sqlite3.connect(str(db_path or DB_PATH))\n    conn.execute(\'PRAGMA foreign_keys = ON\')\n    conn.row_factory = sqlite3.Row\n    return conn\n\n\ndef init_database(db_path=None):\n    with get_connection(db_path) as conn:\n        conn.executescript(\'\'\'\n        CREATE TABLE IF NOT EXISTS books (\n            id INTEGER PRIMARY KEY AUTOINCREMENT,\n            title TEXT NOT NULL CHECK(length(trim(title)) > 0),\n            author TEXT NOT NULL CHECK(length(trim(author)) > 0),\n            year INTEGER CHECK(year BETWEEN 1000 AND 2100),\n            quantity INTEGER NOT NULL DEFAULT 1 CHECK(quantity > 0)\n        );\n        CREATE TABLE IF NOT EXISTS readers (\n            id INTEGER PRIMARY KEY AUTOINCREMENT,\n            last_name TEXT NOT NULL,\n            first_name TEXT NOT NULL,\n            middle_name TEXT,\n            student_card TEXT NOT NULL UNIQUE,\n            group_name TEXT,\n            phone TEXT,\n            email TEXT\n        );\n        CREATE TABLE IF NOT EXISTS issues (\n            id INTEGER PRIMARY KEY AUTOINCREMENT,\n            book_id INTEGER NOT NULL REFERENCES books(id),\n            reader_id INTEGER NOT NULL REFERENCES readers(id),\n            issue_date TEXT NOT NULL,\n            due_date TEXT NOT NULL,\n            return_date TEXT,\n            status TEXT NOT NULL DEFAULT \'выдана\'\n                CHECK(status IN (\'выдана\', \'возвращена\'))\n        );\n        \'\'\')\n\n\ndef add_book(title, author, year, quantity=1, db_path=None):\n    from library_functions import validate_year, validate_quantity\n    if not title or not title.strip() or not author or not author.strip():\n        raise ValueError(\'Название и автор обязательны\')\n    validate_year(year)\n    validate_quantity(quantity)\n    with get_connection(db_path) as conn:\n        return conn.execute(\'INSERT INTO books(title,author,year,quantity) VALUES(?,?,?,?)\',\n                            (title.strip(), author.strip(), year, quantity)).lastrowid\n\n\ndef add_reader(last_name, first_name, student_card, group_name=\'\', phone=\'\', db_path=None):\n    from library_functions import validate_student_card\n    validate_student_card(student_card)\n    if not last_name.strip() or not first_name.strip():\n        raise ValueError(\'Имя и фамилия обязательны\')\n    with get_connection(db_path) as conn:\n        return conn.execute(\'\'\'INSERT INTO readers(last_name,first_name,student_card,group_name,phone)\n                              VALUES(?,?,?,?,?)\'\'\',\n                            (last_name.strip(), first_name.strip(), student_card.strip(), group_name, phone)).lastrowid\n\n\ndef search_books(query=\'\', db_path=None):\n    with get_connection(db_path) as conn:\n        return [dict(row) for row in conn.execute(\n            \'SELECT * FROM books WHERE title LIKE ? OR author LIKE ? ORDER BY title\',\n            (f\'%{query}%\', f\'%{query}%\'))]\n\n\ndef list_readers(db_path=None):\n    with get_connection(db_path) as conn:\n        return [dict(row) for row in conn.execute(\'SELECT * FROM readers ORDER BY last_name,first_name\')]\n\n\n# Functions for labs 3–6. All SQL statements use parameters.\nfrom datetime import date, timedelta\nimport csv\nimport shutil\n\n\ndef get_all_books(db_path=None):\n    with get_connection(db_path) as conn:\n        return [dict(r) for r in conn.execute(\'SELECT * FROM books ORDER BY id\')]\n\n\ndef get_books_by_author(author, db_path=None):\n    with get_connection(db_path) as conn:\n        return [dict(r) for r in conn.execute(\'SELECT * FROM books WHERE author LIKE ? ORDER BY title\', (\'%\' + author + \'%\',))]\n\n\ndef get_all_readers(db_path=None):\n    return list_readers(db_path)\n\n\ndef get_all_issues(db_path=None):\n    with get_connection(db_path) as conn:\n        return [dict(r) for r in conn.execute("""SELECT i.*, b.title AS book_title,\n             r.last_name || \' \' || r.first_name AS reader_name\n             FROM issues i JOIN books b ON b.id=i.book_id\n             JOIN readers r ON r.id=i.reader_id ORDER BY i.id DESC""")]\n\n\ndef edit_book(book_id, title, author, year, quantity, db_path=None):\n    from library_functions import validate_year, validate_quantity\n    validate_year(year); validate_quantity(quantity)\n    if not title.strip() or not author.strip(): raise ValueError(\'Укажите название и автора\')\n    with get_connection(db_path) as conn:\n        active = conn.execute("SELECT COUNT(*) FROM issues WHERE book_id=? AND return_date IS NULL", (book_id,)).fetchone()[0]\n        if quantity < active: raise ValueError(\'Количество меньше числа выданных экземпляров\')\n        return conn.execute(\'UPDATE books SET title=?,author=?,year=?,quantity=? WHERE id=?\',\n                            (title.strip(),author.strip(),year,quantity,book_id)).rowcount\n\n\ndef delete_book(book_id, db_path=None):\n    with get_connection(db_path) as conn:\n        return conn.execute(\'DELETE FROM books WHERE id=?\', (book_id,)).rowcount\n\n\ndef issue_book(book_id, reader_id, days=14, db_path=None, issue_date=None):\n    if type(days) is not int or days < 1: raise ValueError(\'Срок выдачи должен быть положительным\')\n    issued = date.fromisoformat(issue_date) if issue_date else date.today()\n    with get_connection(db_path) as conn:\n        book=conn.execute(\'SELECT quantity FROM books WHERE id=?\',(book_id,)).fetchone()\n        if book is None: raise ValueError(\'Книга не найдена\')\n        if conn.execute(\'SELECT 1 FROM readers WHERE id=?\',(reader_id,)).fetchone() is None:\n            raise ValueError(\'Читатель не найден\')\n        active=conn.execute(\'SELECT COUNT(*) FROM issues WHERE book_id=? AND return_date IS NULL\',(book_id,)).fetchone()[0]\n        if active >= book[\'quantity\']: raise ValueError(\'Нет доступных экземпляров\')\n        return conn.execute("""INSERT INTO issues(book_id,reader_id,issue_date,due_date,status)\n            VALUES(?,?,?,?,?)""",(book_id,reader_id,issued.isoformat(),(issued+timedelta(days=days)).isoformat(),\'выдана\')).lastrowid\n\n\ndef return_book(issue_id, db_path=None, return_date=None):\n    returned=date.fromisoformat(return_date) if return_date else date.today()\n    with get_connection(db_path) as conn:\n        issue=conn.execute(\'SELECT issue_date,return_date FROM issues WHERE id=?\',(issue_id,)).fetchone()\n        if issue is None: raise ValueError(\'Выдача не найдена\')\n        if issue[\'return_date\'] is not None: raise ValueError(\'Книга уже возвращена\')\n        if returned < date.fromisoformat(issue[\'issue_date\']): raise ValueError(\'Возврат раньше выдачи\')\n        conn.execute("UPDATE issues SET return_date=?,status=\'возвращена\' WHERE id=?",(returned.isoformat(),issue_id))\n        return True\n\n\ndef edit_reader(reader_id, last_name, first_name, student_card, group_name=\'\', phone=\'\', db_path=None):\n    from library_functions import validate_student_card\n    validate_student_card(student_card)\n    if not last_name.strip() or not first_name.strip(): raise ValueError(\'Имя и фамилия обязательны\')\n    with get_connection(db_path) as conn:\n        return conn.execute("""UPDATE readers SET last_name=?,first_name=?,student_card=?,group_name=?,phone=? WHERE id=?""",\n                            (last_name.strip(),first_name.strip(),student_card,group_name,phone,reader_id)).rowcount\n\n\ndef delete_reader(reader_id, db_path=None):\n    with get_connection(db_path) as conn:\n        return conn.execute(\'DELETE FROM readers WHERE id=?\',(reader_id,)).rowcount\n\n\ndef get_debtors(db_path=None, as_of=None):\n    today=as_of or date.today().isoformat()\n    with get_connection(db_path) as conn:\n        return [dict(r) for r in conn.execute("""SELECT i.id AS issue_id, r.id AS reader_id,\n            r.last_name,r.first_name,r.student_card,b.title,i.due_date,\n            CAST(julianday(?) - julianday(i.due_date) AS INTEGER) AS overdue_days\n            FROM issues i JOIN readers r ON i.reader_id=r.id JOIN books b ON i.book_id=b.id\n            WHERE i.return_date IS NULL AND i.due_date < ? ORDER BY i.due_date""",(today,today))]\n\n\ndef get_most_popular_books(limit=10, db_path=None):\n    with get_connection(db_path) as conn:\n        return [dict(r) for r in conn.execute("""SELECT b.id,b.title,b.author,COUNT(i.id) AS issue_count\n            FROM books b LEFT JOIN issues i ON i.book_id=b.id\n            GROUP BY b.id ORDER BY issue_count DESC,b.title LIMIT ?""",(limit,))]\n\n\ndef get_book_history(book_id, db_path=None):\n    with get_connection(db_path) as conn:\n        return [dict(r) for r in conn.execute("""SELECT i.*,r.last_name,r.first_name\n          FROM issues i JOIN readers r ON i.reader_id=r.id\n          WHERE i.book_id=? ORDER BY i.issue_date DESC""",(book_id,))]\n\n\ndef get_library_stats(db_path=None):\n    with get_connection(db_path) as conn:\n        q=lambda sql:conn.execute(sql).fetchone()[0]\n        return dict(books=q(\'SELECT COUNT(*) FROM books\'), copies=q(\'SELECT COALESCE(SUM(quantity),0) FROM books\'),\n          readers=q(\'SELECT COUNT(*) FROM readers\'), issues_total=q(\'SELECT COUNT(*) FROM issues\'),\n          issues_active=q(\'SELECT COUNT(*) FROM issues WHERE return_date IS NULL\'),\n          issues_returned=q(\'SELECT COUNT(*) FROM issues WHERE return_date IS NOT NULL\'))\n\n\ndef get_active_readers(limit=10, db_path=None):\n    with get_connection(db_path) as conn:\n        return [dict(r) for r in conn.execute("""SELECT r.id,r.last_name,r.first_name,r.student_card,\n         COUNT(i.id) AS issue_count FROM readers r LEFT JOIN issues i ON i.reader_id=r.id\n         GROUP BY r.id ORDER BY issue_count DESC,r.last_name LIMIT ?""",(limit,))]\n\n\ndef export_books_to_csv(filename=\'books.csv\', db_path=None):\n    rows=get_all_books(db_path)\n    with open(filename,\'w\',encoding=\'utf-8-sig\',newline=\'\') as file:\n        writer=csv.DictWriter(file,fieldnames=[\'id\',\'title\',\'author\',\'year\',\'quantity\']); writer.writeheader(); writer.writerows(rows)\n    return filename\n\n\ndef export_debtors_to_txt(filename=\'debtors.txt\', db_path=None, as_of=None):\n    rows=get_debtors(db_path,as_of)\n    with open(filename,\'w\',encoding=\'utf-8\') as file:\n        file.write(\'Читатели с просроченными книгами\\n\')\n        if not rows: file.write(\'Просроченных выдач нет\\n\')\n        for r in rows: file.write(f"{r[\'last_name\']} {r[\'first_name\']} | {r[\'title\']} | до {r[\'due_date\']} | дней: {r[\'overdue_days\']}\\n")\n    return filename\n\n\ndef backup_database(filename=\'library_backup.db\', db_path=None):\n    # SQLite backup API provides a consistent snapshot even when source is open.\n    with get_connection(db_path) as source:\n        with sqlite3.connect(str(filename)) as target: source.backup(target)\n    return filename\n', encoding="utf-8")
print("Создан database.py")


In [ ]:
from pathlib import Path
Path('seed.py').write_text('"""Демонстрационные данные, повторный запуск не создаёт дубликаты."""\nfrom datetime import date,timedelta\nfrom database import *\n\ndef seed(db_path=None):\n    init_database(db_path)\n    with get_connection(db_path) as conn:\n        if conn.execute(\'SELECT COUNT(*) FROM books\').fetchone()[0]: return\n    b1=add_book(\'Война и мир\',\'Лев Толстой\',1869,3,db_path)\n    b2=add_book(\'Преступление и наказание\',\'Фёдор Достоевский\',1866,2,db_path)\n    b3=add_book(\'Мастер и Маргарита\',\'Михаил Булгаков\',1967,4,db_path)\n    r1=add_reader(\'Иванов\',\'Иван\',\'СБ-001\',\'3ИСП4\',\'+7-900-000-00-01\',db_path)\n    r2=add_reader(\'Петрова\',\'Анна\',\'СБ-002\',\'3ИСП4\',\'+7-900-000-00-02\',db_path)\n    r3=add_reader(\'Сидоров\',\'Максим\',\'СБ-003\',\'3ИСП4\',\'+7-900-000-00-03\',db_path)\n    issue_book(b1,r1,7,db_path,issue_date=(date.today()-timedelta(days=30)).isoformat())\n    i=issue_book(b2,r2,14,db_path,issue_date=(date.today()-timedelta(days=20)).isoformat())\n    return_book(i,db_path,return_date=(date.today()-timedelta(days=10)).isoformat())\n    issue_book(b3,r3,14,db_path)\n\nif __name__==\'__main__\':\n    seed(); print(\'Демонстрационная база готова\')\n', encoding="utf-8")
print("Создан seed.py")


In [ ]:
from pathlib import Path
Path('app.py').write_text('"""Консольная информационная система студенческой библиотеки."""\nimport sqlite3\nimport database as db\n\nMENU = """\nСТУДЕНЧЕСКАЯ БИБЛИОТЕКА\n 1 Книги                 2 Поиск по автору\n 3 Читатели              4 Все выдачи\n 5 Добавить книгу        6 Изменить книгу       7 Удалить книгу\n 8 Выдать книгу          9 Вернуть книгу\n10 Добавить читателя    11 Изменить читателя  12 Удалить читателя\n13 Должники             14 Популярные книги   15 История книги\n16 Статистика           17 Активные читатели\n18 Экспорт книг CSV     19 Экспорт должников TXT  20 Резервная копия\n 0 Выход\n"""\n\ndef number(label): return int(input(label + \': \'))\ndef show(rows):\n    for row in rows: print(\' | \'.join(f\'{k}: {v}\' for k,v in row.items()))\n    if not rows: print(\'Записей нет\')\ndef book_fields():\n    return input(\'Название: \'), input(\'Автор: \'), number(\'Год\'), number(\'Количество\')\ndef reader_fields():\n    return input(\'Фамилия: \'), input(\'Имя: \'), input(\'Студбилет: \'), input(\'Группа: \'), input(\'Телефон: \')\n\ndef main():\n    db.init_database()\n    while True:\n        print(MENU); choice=input(\'Выбор: \').strip()\n        try:\n            if choice==\'0\': return\n            elif choice==\'1\': show(db.get_all_books())\n            elif choice==\'2\': show(db.get_books_by_author(input(\'Автор: \')))\n            elif choice==\'3\': show(db.get_all_readers())\n            elif choice==\'4\': show(db.get_all_issues())\n            elif choice==\'5\': print(\'ID:\',db.add_book(*book_fields()))\n            elif choice==\'6\': print(\'Изменено:\',db.edit_book(number(\'ID книги\'),*book_fields()))\n            elif choice==\'7\': print(\'Удалено:\',db.delete_book(number(\'ID книги\')))\n            elif choice==\'8\': print(\'ID выдачи:\',db.issue_book(number(\'ID книги\'),number(\'ID читателя\'),number(\'Срок в днях\')))\n            elif choice==\'9\': print(\'Возвращено:\',db.return_book(number(\'ID выдачи\')))\n            elif choice==\'10\': print(\'ID:\',db.add_reader(*reader_fields()))\n            elif choice==\'11\': print(\'Изменено:\',db.edit_reader(number(\'ID читателя\'),*reader_fields()))\n            elif choice==\'12\': print(\'Удалено:\',db.delete_reader(number(\'ID читателя\')))\n            elif choice==\'13\': show(db.get_debtors())\n            elif choice==\'14\': show(db.get_most_popular_books())\n            elif choice==\'15\': show(db.get_book_history(number(\'ID книги\')))\n            elif choice==\'16\': print(db.get_library_stats())\n            elif choice==\'17\': show(db.get_active_readers())\n            elif choice==\'18\': print(\'Сохранено:\',db.export_books_to_csv())\n            elif choice==\'19\': print(\'Сохранено:\',db.export_debtors_to_txt())\n            elif choice==\'20\': print(\'Сохранено:\',db.backup_database())\n            else: print(\'Неизвестный пункт\')\n        except (ValueError, sqlite3.Error) as e: print(\'Ошибка:\',e)\n\nif __name__==\'__main__\': main()\n', encoding="utf-8")
print("Создан app.py")


## Создание базы данных и примеры выборок

In [ ]:
from seed import seed
seed()
from database import *
print("Книги:",get_all_books())
print("Читатели:",get_all_readers())
print("Выдачи:",get_all_issues())


In [ ]:
print("Поиск Толстого:",get_books_by_author("Толстой"))
print("Должники:",get_debtors())
print("Популярные книги:",get_most_popular_books())
print("Активные читатели:",get_active_readers())
print("Статистика:",get_library_stats())


## Экспорт CSV/TXT и резервное копирование

In [ ]:
print(export_books_to_csv())
print(export_debtors_to_txt())
print(backup_database())


## Тестирование unittest / pytest и покрытие кода

In [ ]:
from pathlib import Path
Path("tests").mkdir(exist_ok=True)
Path("tests/test_database.py").write_text("import tempfile\nimport unittest\nfrom pathlib import Path\nfrom database import init_database, add_book, add_reader, search_books, list_readers\n\nclass TestDatabase(unittest.TestCase):\n    def setUp(self):\n        self.temp = tempfile.TemporaryDirectory()\n        self.db_path = Path(self.temp.name) / 'test.db'\n        init_database(self.db_path)\n\n    def tearDown(self):\n        self.temp.cleanup()\n\n    def test_add_and_find_book(self):\n        add_book('Война и мир', 'Лев Толстой', 1869, 3, self.db_path)\n        self.assertEqual(len(search_books('Толстой', self.db_path)), 1)\n\n    def test_add_reader(self):\n        add_reader('Иванов', 'Иван', 'СБ-001', db_path=self.db_path)\n        self.assertEqual(len(list_readers(self.db_path)), 1)\n\n    def test_empty_search(self):\n        self.assertEqual(search_books('нет', self.db_path), [])\n\nif __name__ == '__main__':\n    unittest.main()\n",encoding="utf-8")


In [ ]:
from pathlib import Path
Path("tests").mkdir(exist_ok=True)
Path("tests/test_library_functions.py").write_text("import pytest\nfrom library_functions import validate_year, validate_quantity, validate_student_card, calculate_overdue_days, format_book_title\n\n@pytest.mark.parametrize('year', [1000, 1869, 2026, 2100])\ndef test_year_valid(year):\n    assert validate_year(year)\n\n@pytest.mark.parametrize('year', [999, 2101, '2000', None, True])\ndef test_year_invalid(year):\n    with pytest.raises(ValueError):\n        validate_year(year)\n\n@pytest.mark.parametrize('quantity', [0, -1, 1.5, '2', True])\ndef test_quantity_invalid(quantity):\n    with pytest.raises(ValueError):\n        validate_quantity(quantity)\n\ndef test_card_empty():\n    with pytest.raises(ValueError):\n        validate_student_card('  ')\n\ndef test_overdue():\n    assert calculate_overdue_days('2026-10-01', '2026-10-06') == 5\n    assert calculate_overdue_days('2026-10-01', '2026-09-30') == 0\n\ndef test_title_truncation():\n    assert format_book_title('А' * 40, 30) == 'А' * 27 + '...'\n",encoding="utf-8")


In [ ]:
from pathlib import Path
Path("tests").mkdir(exist_ok=True)
Path("tests/test_workflow.py").write_text("import tempfile\nfrom pathlib import Path\nimport pytest\nfrom database import *\n\n@pytest.fixture\ndef dbfile(tmp_path):\n    p=tmp_path/'test.db'; init_database(p); return p\n\ndef test_full_workflow(dbfile,tmp_path):\n    b=add_book('Тест','Автор',2020,1,dbfile)\n    r=add_reader('Иванов','Иван','ABC-001',db_path=dbfile)\n    assert get_books_by_author('Авт',dbfile)[0]['id']==b\n    assert len(get_all_books(dbfile))==1\n    assert len(get_all_readers(dbfile))==1\n    i=issue_book(b,r,7,dbfile,issue_date='2024-01-01')\n    assert len(get_all_issues(dbfile))==1\n    assert get_debtors(dbfile,'2024-02-01')[0]['overdue_days']==24\n    with pytest.raises(ValueError): issue_book(b,r,7,dbfile)\n    assert get_most_popular_books(db_path=dbfile)[0]['issue_count']==1\n    assert get_book_history(b,dbfile)[0]['id']==i\n    assert get_active_readers(db_path=dbfile)[0]['issue_count']==1\n    assert get_library_stats(dbfile)['issues_active']==1\n    assert return_book(i,dbfile,'2024-01-09')\n    assert get_library_stats(dbfile)['issues_returned']==1\n    assert get_debtors(dbfile,'2024-02-01')==[]\n    assert edit_book(b,'Тест2','Автор2',2021,2,dbfile)==1\n    assert edit_reader(r,'Петров','Пётр','ABC-002',db_path=dbfile)==1\n    assert Path(export_books_to_csv(tmp_path/'books.csv',dbfile)).exists()\n    assert Path(export_debtors_to_txt(tmp_path/'debtors.txt',dbfile)).exists()\n    backup_database(tmp_path/'backup.db',dbfile)\n    with get_connection(tmp_path/'backup.db') as conn:\n        assert conn.execute('SELECT COUNT(*) FROM books').fetchone()[0]==1\n    with pytest.raises(Exception): delete_book(b,dbfile)  # history protected by FK\n\ndef test_validation_and_deletion(dbfile):\n    b=add_book('Тест','Автор',2020,2,dbfile)\n    r=add_reader('Тестов','Тест','X-1',db_path=dbfile)\n    assert delete_book(b,dbfile)==1\n    assert delete_reader(r,dbfile)==1\n    with pytest.raises(ValueError): add_book('','Автор',2020,1,dbfile)\n",encoding="utf-8")


In [ ]:
%pip -q install pytest pytest-cov
!python -m unittest discover -s tests -p "test_database.py" -v
!python -m pytest -q
!python -m pytest --cov=database --cov=library_functions --cov-report=term-missing -q


## Скачать результаты

Файлы можно скачать через левую панель **Files**. Для сохранения в Google Drive выберите «Файл → Сохранить копию на Диске».